In [1]:
import json
import os

import re
from concurrent.futures import ThreadPoolExecutor
from datetime import date, timedelta
from typing import Literal

import requests
from openai import OpenAI
from pydantic import BaseModel, Field

In [2]:
MODEL = "gpt-6-luna"

SYSTEM_PROMPT = """
You are a professional AI career coach. Help users prepare for interviews, improve their CVs, and make career decisions.

On the first turn, introduce yourself briefly and ask which service the user wants:
1. Mock interview : if the user choose this you should fist ask about the level 
2. CV review
3. Career advice
Do not start a service until the user chooses one. Return status "needs_mode" with mode null, and put the introduction and question in message.

When the user chooses a service, return status "selected" and its mode: "mock_interview", "cv_review", or "career_advice". If their choice is unclear, ask them to choose and return status "needs_mode".
Once a service is selected, keep that mode for follow-up messages unless the user clearly asks to switch services.

In mock interview mode, ask one interview question at a time. When the interview is complete, provide a score, strengths, weaknesses, and actionable feedback.

In CV review mode, review the supplied CV for the target role. Give a concise summary, strengths, and concrete improvements. Ask for the CV or target role if missing.

In career advice mode, give practical, personalized guidance with actionable next steps. Ask for any important missing context.
"""

In [3]:
class ModeSelection(BaseModel):
    status: Literal["needs_mode", "selected"]
    mode: Literal["mock_interview", "cv_review", "career_advice"] | None = None
    message: str


class MockInterviewReply(BaseModel):
    status: Literal["needs_info", "in_progress", "complete"]
    message: str
    next_question: str | None = None
    feedback: str | None = None


class CVReviewReply(BaseModel):
    status: Literal["needs_info", "complete"]
    summary: str
    strengths: list[str]
    improvements: list[str]


class CareerAdviceReply(BaseModel):
    status: Literal["needs_info", "complete"]
    message: str
    action_steps: list[str]


TOOLS = []


class CareerCoach:
    """Creates a career coach."""

    def __init__(self) -> None:
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is missing")

        self.client = OpenAI(api_key=api_key, timeout=60.0, max_retries=0)

    def _request(
        self,
        input_data: str,
        output_model: type[BaseModel],
        previous_response_id: str | None = None,
    ):
        request = {
            "model": MODEL,
            "reasoning": {"effort": "low"},
            "instructions": SYSTEM_PROMPT,
            "input": input_data,
            "text_format": output_model,
            "tools": TOOLS,
        }
        if previous_response_id:
            request["previous_response_id"] = previous_response_id
        return self.client.responses.parse(**request)

    def get_response(
        self,
        user_message: str,
        previous_response_id: str | None = None,
    ) -> dict[str, object]:
        
        selection_response = self._request(
            user_message, ModeSelection, previous_response_id
        )
        selection = selection_response.output_parsed
        if selection is None:
            raise ValueError("The model did not return a parsed mode selection")

        if selection.status == "needs_mode" or selection.mode is None:
            return {
                "response_id": selection_response.id,
                "mode": None,
                "reply": selection,
            }

        mode_models = {
            "mock_interview": MockInterviewReply,
            "cv_review": CVReviewReply,
            "career_advice": CareerAdviceReply,
        }
        reply_response = self._request(
            f"Continue the conversation in {selection.mode} mode. Respond to the user's latest message.",
            mode_models[selection.mode],
            selection_response.id,
        )
        return {
            "response_id": reply_response.id,
            "mode": selection.mode,
            "reply": reply_response.output_parsed,
        }

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

env_path = os.path.join(os.getcwd(), ".env")
load_dotenv(env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(f"OPENAI_API_KEY was not found. Checked: {env_path}")



In [5]:
coach = CareerCoach()
result = coach.get_response("Hi!")

print(result["reply"].message)

Hi! I’m your AI career coach. What would you like help with?
1. Mock interview
2. CV review
3. Career advice


In [6]:
result = coach.get_response(
    "I would like a mock interview for a data analyst role.",
    result["response_id"])

print(result["reply"].status)
print(result["reply"].message)

needs_info
What level should I tailor the data analyst mock interview to—entry-level, mid-level, or senior?


In [7]:
result = coach.get_response(
    "mid-level",
    result["response_id"],
)

print(result["reply"].status)
print(result["reply"].message)


in_progress
Please answer the interview question above: describe a time you used data to influence a business decision, including the problem, your analysis, and the outcome.
